In [1]:
!pip install stable-baselines3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 184.5/184.5 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 26.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 65.4 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalli

In [2]:
!pip install finrl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.2/127.2 kB 1.7 MB/s eta 0:00:00


In [3]:
from google.colab import drive
import pandas as pd
import os

# Mount Google Drive
drive.mount('/content/drive')

# Define the directory path
dir_path = "/content/drive/MyDrive/sp500_csv"

# Get the list of files in the directory
file_list = os.listdir(dir_path)

# Assuming you want to read the first CSV file in the directory:
file_path = os.path.join(dir_path, file_list[0])

# Load CSV into DataFrame
df = pd.read_csv(file_path)
df.head()

Mounted at /content/drive


,Date,Low,Open,Volume,High,Close,Adjusted Close
0,04-04-1995,2.81250,2.93750,40387200,2.96875,2.953125,2.275675
1,05-04-1995,2.84375,2.90625,12236800,2.96875,2.843750,2.191389
2,06-04-1995,2.84375,2.84375,3776000,2.93750,2.890625,2.227511
3,07-04-1995,2.84375,2.90625,1920800,2.90625,2.843750,2.191389
4,10-04-1995,2.84375,2.84375,2047200,2.90625,2.875000,2.215471


In [4]:
!pip install shimmy>=2.0

In [15]:
import gym
import numpy as np
import pandas as pd
from stable_baselines3 import DDPG
from stable_baselines3.common.noise import NormalActionNoise
from stable_baselines3.common.evaluation import evaluate_policy
from gym import spaces

class StockTradingEnv(gym.Env):
    def __init__(self, df, initial_balance=10000, max_steps=500):
        super(StockTradingEnv, self).__init__()
        self.df = df
        self.initial_balance = initial_balance
        self.current_step = 0
        self.max_steps = min(len(df) - 1, max_steps)

        self.action_space = spaces.Box(low=-1, high=1, shape=(1,), dtype=np.float32)
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(6,), dtype=np.float32)
        self.reset()

    def reset(self):
        self.current_step = 0
        self.balance = self.initial_balance
        return self._next_observation()

    def step(self, action):
        self.current_step += 1
        done = self.current_step >= self.max_steps
        return self._next_observation(), 0, done, {}

    def _next_observation(self):
        return np.random.random(6)


# Create Environment
env = StockTradingEnv(df)

# Define Model with Noise
n_actions = env.action_space.shape[-1]
action_noise = NormalActionNoise(mean=np.zeros(n_actions), sigma=0.1 * np.ones(n_actions))

model = DDPG("MlpPolicy", env, action_noise=action_noise, verbose=1)

# Train Model
model.learn(total_timesteps=50000)

# Save Model
model.save("ddpg_stock_trader")

# Load Model for Prediction
model = DDPG.load("ddpg_stock_trader")

def predict_stock_action(obs):
    action, _ = model.predict(obs, deterministic=True)
    return action

# Test Prediction
obs = env.reset()
predicted_action = predict_stock_action(obs)
print("Predicted Action:", predicted_action)


Using cpu device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


/usr/local/lib/python3.11/dist-packages/stable_baselines3/common/vec_env/patch_gym.py:49: UserWarning: You provided an OpenAI Gym environment. We strongly recommend transitioning to Gymnasium environments. Stable-Baselines3 is automatically wrapping your environments in a compatibility layer, which could potentially cause issues.
  warnings.warn(


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 500      |
|    ep_rew_mean     | 0        |
| time/              |          |
|    episodes        | 4        |
|    fps             | 36       |
|    time_elapsed    | 54       |
|    total_timesteps | 2000     |
| train/             |          |
|    actor_loss      | -0.0461  |
|    critic_loss     | 1.58e-06 |
|    learning_rate   | 0.001    |
|    n_updates       | 1899     |
---------------------------------
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 500      |
|    ep_rew_mean     | 0        |
| time/              |          |
|    episodes        | 8        |
|    fps             | 35       |
|    time_elapsed    | 112      |
|    total_timesteps | 4000     |
| train/             |          |
|    actor_loss      | -0.0426  |
|    critic_loss     | 2.56e-06 |
|    learning_rate   | 0.001    |
|    n_updates       | 3899     |
--------------

In [16]:
import numpy as np
import pandas as pd
from scipy.stats import norm

# Load your data (adjust file path if needed)
# Mount Google Drive
drive.mount('/content/drive')

# Define the directory path
dir_path = "/content/drive/MyDrive/sp500_csv"

# Get the list of files in the directory
file_list = os.listdir(dir_path)

# Assuming you want to read the first CSV file in the directory:
file_path = os.path.join(dir_path, file_list[0])

# Load CSV into DataFrame
df = pd.read_csv(file_path)

# Preprocess data: Convert dates and calculate returns
df['Date'] = pd.to_datetime(df['Date'], format='%d-%m-%Y')  # Ensure 'Date' column exists (adjust if named differently)
df = df.sort_values('Date')  # Sort chronologically
df['returns'] = df['Close'].pct_change()  # Daily returns
df = df.dropna()  # Remove first row (NaN return)

def calculate_performance_metrics(df, risk_free_rate=0.0, trading_days=252):
    """Calculate key performance metrics for a trading strategy."""
    daily_returns = df['returns']
    n_days = len(daily_returns)

    # Cumulative and Annualized Returns
    cumulative_returns = (1 + daily_returns).prod() - 1
    annual_return = (1 + cumulative_returns) ** (trading_days/n_days) - 1

    # Volatility and Risk-Adjusted Metrics
    annual_volatility = daily_returns.std() * np.sqrt(trading_days)
    sharpe_ratio = (annual_return - risk_free_rate) / annual_volatility

    # Drawdown Analysis
    cumulative = (1 + daily_returns).cumprod()
    peak = cumulative.expanding().max()
    drawdown = (cumulative - peak) / peak
    max_drawdown = drawdown.min()
    calmar_ratio = annual_return / abs(max_drawdown) if max_drawdown != 0 else np.nan

    # Stability (R² of cumulative returns trend)
    x = np.arange(n_days)
    y = cumulative.values
    coef = np.polyfit(x, y, 1)
    y_pred = np.polyval(coef, x)
    ss_res = ((y - y_pred) ** 2).sum()
    ss_tot = ((y - y.mean()) ** 2).sum()
    stability = 1 - (ss_res / ss_tot) if ss_tot != 0 else np.nan

    # Omega and Sortino Ratios
    threshold = risk_free_rate / trading_days
    excess_returns = daily_returns - threshold
    omega_ratio = (excess_returns[excess_returns > 0].sum() /
                   abs(excess_returns[excess_returns < 0].sum())) if excess_returns[excess_returns < 0].sum() != 0 else np.nan
    downside_volatility = daily_returns[daily_returns < threshold].std() * np.sqrt(trading_days)
    sortino_ratio = (annual_return - risk_free_rate) / downside_volatility if downside_volatility != 0 else np.nan

    # Higher Moments and Tail Risk
    skew = daily_returns.skew() if n_days > 2 else np.nan
    kurtosis = daily_returns.kurtosis() if n_days > 3 else np.nan
    tail_ratio = abs(daily_returns.quantile(0.95)) / abs(daily_returns.quantile(0.05)) if daily_returns.quantile(0.05) != 0 else np.nan
    daily_var = daily_returns.quantile(0.05)

    return pd.Series({
        'Annual return': annual_return,
        'Cumulative returns': cumulative_returns,
        'Annual volatility': annual_volatility,
        'Sharpe ratio': sharpe_ratio,
        'Calmar ratio': calmar_ratio,
        'Stability': stability,
        'Max drawdown': max_drawdown,
        'Omega ratio': omega_ratio,
        'Sortino ratio': sortino_ratio,
        'Skew': skew,
        'Kurtosis': kurtosis,
        'Tail ratio': tail_ratio,
        'Daily value at risk': daily_var
    })

# Usage
metrics = calculate_performance_metrics(df)
print(metrics)
print(f"\nShape of DataFrame: {df.shape}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Annual return           0.156784
Cumulative returns     55.231957
Annual volatility       0.326581
Sharpe ratio            0.480078
Calmar ratio            0.220792
Stability               0.833635
Max drawdown           -0.710100
Omega ratio             1.123231
Sortino ratio           0.643206
Skew                   -0.149513
Kurtosis               14.758922
Tail ratio              1.011239
Daily value at risk    -0.029885
dtype: float64

Shape of DataFrame: (6972, 8)
